# Data Exploration and Experimentation Notebook

This notebook was used throughout the project for testing, exploring, and experimenting with the dataset. It contains a variety of analyses and visualizations, including:
- Initial data inspection and cleaning
- Feature selection and correlation analysis
- Distribution and value count visualizations for each feature
- Experimentation with transformations and preprocessing techniques
- Testing out ideas and approaches before integrating them into the main pipeline

The notebook serves as a sandbox for understanding the data, identifying issues, and developing preprocessing strategies. It is not intended as a final report, but as a working document to support the data mining process.

In [1]:
# Imports
import os

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import QuantileTransformer

In [ ]:
data = pd.read_csv("data.csv")
# benign = pd.read_csv("new_dataset/benign_normalized.csv")
# malicious = pd.read_csv("new_dataset/malicious_normalized.csv")
# data = pd.concat([benign, malicious], ignore_index=True)

In [ ]:
# drop constant value cols
unique_per_col = data.nunique()
print(data.shape)
constant_cols = [
    unique_per_col.axes[0][i]
    for i, unique_count in enumerate(unique_per_col)
    if unique_count == 1
]

print(constant_cols)
data = data.drop(columns=constant_cols)
print(data.shape)

(8656767, 86)
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg']
(8656767, 81)


In [3]:
# find correlations
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
corr = data[numeric_cols].corr()

step = 0
all_to_drop = []

while True:
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    # Find all pairs with abs(corr) > 0.9
    pairs = np.where(upper.abs() > 0.9)
    if len(pairs[0]) == 0:
        break
    to_drop = dict()
    print(f"\n=== Step {step} ===")
    for i, j in zip(pairs[0], pairs[1]):
        col1 = upper.index[i]
        col2 = upper.columns[j]
        var1 = data[col1].var()
        var2 = data[col2].var()
        drop_col = col1 if var1 < var2 else col2
        correlated_col = col2 if drop_col == col1 else col1
        # Only keep the first correlated column for each drop_col in this step
        if drop_col not in to_drop:
            to_drop[drop_col] = correlated_col
            print(f"Removing column due to high correlation and lower variance: {drop_col} (correlated with: {correlated_col})")
            all_to_drop.append((drop_col, correlated_col))
    corr = corr.drop(columns=to_drop.keys(), errors='ignore').drop(index=to_drop.keys(), errors='ignore')
    step += 1

# Write to txt file
with open("correlations.txt", "w", encoding="utf-8") as f:
    for drop_col, correlated_col in all_to_drop:
        f.write(f"'{drop_col}',                 #λογω correlation με {correlated_col}\n")



=== Step 0 ===
Removing column due to high correlation and lower variance: Fwd IAT Total (correlated with: Flow Duration)
Removing column due to high correlation and lower variance: Subflow Fwd Packets (correlated with: Total Fwd Packet)
Removing column due to high correlation and lower variance: Subflow Bwd Packets (correlated with: Total Bwd packets)
Removing column due to high correlation and lower variance: Fwd Act Data Pkts (correlated with: Total Length of Fwd Packet)
Removing column due to high correlation and lower variance: Fwd Packet Length Min (correlated with: Fwd Packet Length Max)
Removing column due to high correlation and lower variance: Fwd Packet Length Mean (correlated with: Fwd Packet Length Max)
Removing column due to high correlation and lower variance: Fwd Packet Length Max (correlated with: Packet Length Max)
Removing column due to high correlation and lower variance: Packet Length Mean (correlated with: Fwd Packet Length Max)
Removing column due to high correl

In [4]:
# Plot and export the correlation matrix 
numeric_cols = corr.select_dtypes(include=["int64", "float64"]).columns

plt.figure(figsize=(20, 16))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar=True,
    annot_kws={"size": 10},
)
plt.title(f"Correlation Matrix - Step {step} (large)")
plt.tight_layout()
plt.savefig(f"correlation_matrix_step_{step}_large.png")
plt.close()

In [5]:
# drop highly correlated low variance pairs
cols_to_drop = list(to_drop.keys())
# Always drop these two columns
cols_to_drop.append("Flow ID")
cols_to_drop.append("Timestamp")

print(data.shape)
data = data.drop(cols_to_drop, axis=1)
print(data.shape)

(8656767, 81)
(8656767, 52)


In [6]:
# # Check for Perfect Predictors (δεν χρησιμοποιήθηκε)
# perfect_predictors = {}

# for col in data.columns:
#     if col == "Label":
#         continue

#     # Group by column value and count labels
#     grouped = data.groupby(col)["Label"].apply(lambda x: (x == "Benign").mean())

#     # Values where all instances are benign
#     perfect = grouped[grouped == 1.0].index.tolist()
#     if len(perfect) > 0:
#         perfect_predictors[col] = perfect

# print(
#     "Columns where specific values always indicate benign:", perfect_predictors
# )

In [7]:
# # Αφαίρεση στηλών με <1% variance (δεν χρησιμοποιήθηκε)
# selector = VarianceThreshold(threshold=0.01)  
# X_reduced = pd.DataFrame(
#     selector.fit_transform(data[numeric_cols]),
#     columns=data[numeric_cols].columns[selector.get_support()],
#     index=data.index
# )
# removed_cols = set(data[numeric_cols].columns) - set(X_reduced.columns)
# print("Columns removed by VarianceThreshold:", list(removed_cols))

In [8]:
# Analyze the remaining columns
for col in data.columns:
    print(f"\n--- {col} ---")
    print(data[col].describe())
    nunique = data[col].nunique()

    # Setup directory and sanitize name
    os.makedirs("histograms", exist_ok=True)
    col_sanitized = col.replace(" ", "_").replace("/", "_").replace("\\", "_")

    if data[col].dtype == "object" or nunique < 20:
        print("\nValue counts:")
        print(data[col].value_counts())
        ax = data[col].value_counts().plot(kind="bar", color="skyblue", alpha=0.7)
        plt.title(f"Value Counts: {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(f"histograms/{col}_bar.png")
        plt.close()
    else:
        print("\nValue counts:")
        print(data[col].value_counts().head(20))
        plt.hist(data[col].dropna(), bins=50, color="skyblue", alpha=0.7)
        plt.title(f"Histogram: {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(f"histograms/{col_sanitized}_hist.png")
        plt.close()


--- Src IP ---
count          8656767
unique              13
top       192.168.1.70
freq           8540951
Name: Src IP, dtype: object

Value counts:
Src IP
192.168.1.70     8540951
192.168.1.177      42228
192.168.1.244      38234
192.168.1.90       24569
192.168.1.59       10361
136.162.16.0         179
192.168.1.220        101
192.168.1.42          71
192.168.1.3           66
8.6.0.1                4
134.221.96.0           1
136.142.2.3            1
136.142.1.3            1
Name: count, dtype: int64

--- Src Port ---
count    8.656767e+06
mean     2.563313e+04
std      2.005925e+04
min      0.000000e+00
25%      6.366000e+03
50%      2.224000e+04
75%      4.234200e+04
max      6.553500e+04
Name: Src Port, dtype: float64

Value counts:
Src Port
2963.0     181994
1181.0     116021
1424.0      89339
1892.0      88156
2389.0      86708
2397.0      83573
1553.0      74983
3021.0      57598
2759.0      53758
2103.0      53534
1207.0      49364
2942.0      43958
80.0        24325
2323.0  

In [9]:
# Visualize quantile transformation on all numerical columns in the dataset. Shows the effect on the whole dataset for each column.

for col in numeric_cols:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f"Quantile Transformation Comparison for '{col}'", fontsize=16
    )
    col_data = data[col].to_numpy().reshape(-1, 1)

    # Top left: Original distribution (all data)
    axes[0, 0].hist(col_data, bins=100, color="gray", alpha=0.7)
    axes[0, 0].set_title(f"Original ({col}) - All Data")

    # Top right: Transformation graph (original vs quantile, all data)
    quantile_transformer = QuantileTransformer(
        output_distribution="normal", random_state=0
    )
    quantile_data = quantile_transformer.fit_transform(col_data)
    n_points = min(100000, len(col_data))
    axes[0, 1].scatter(
        np.sort(col_data[:n_points, 0]),
        np.sort(quantile_data[:n_points, 0]),
        alpha=0.5,
        s=10,
        color="orange",
    )
    axes[0, 1].set_xlabel(f"Original {col}")
    axes[0, 1].set_ylabel("Quantile Transformed")
    axes[0, 1].set_title("Original vs Quantile (All Data)")

    # Bottom left: Quantile transformed (all data)
    axes[1, 0].hist(quantile_data, bins=100, color="purple", alpha=0.7)
    axes[1, 0].set_title(f"Quantile Transformed - All Data")
    axes[1, 0].set_xlabel(f"Quantile {col}")
    axes[1, 0].set_ylabel("Count")

    # Bottom right: Quantile transformed (without most dominant value)
    col_flat = data[col].dropna().values
    if len(col_flat) > 1:
        vals, counts = np.unique(col_flat, return_counts=True)
        most_freq_val = vals[np.argmax(counts)]
        filtered = col_flat[col_flat != most_freq_val]
        if len(filtered) > 0:
            filtered = filtered.reshape(-1, 1)
            quantile_data_filtered = quantile_transformer.fit_transform(filtered)
            axes[1, 1].hist(quantile_data_filtered, bins=100, color="teal", alpha=0.7)
            axes[1, 1].set_title("Quantile Transformed (w/o Dominant)")
            axes[1, 1].set_xlabel(f"Quantile {col} (w/o dominant)")
            axes[1, 1].set_ylabel("Count")
        else:
            axes[1, 1].text(0.5, 0.5, "Not enough data", ha="center", va="center")
            axes[1, 1].set_title("Quantile Transformed (w/o Dominant)")
    else:
        axes[1, 1].text(0.5, 0.5, "Not enough data", ha="center", va="center")
        axes[1, 1].set_title("Quantile Transformed (w/o Dominant)")

    os.makedirs("transformation_preview", exist_ok=True)
    col_sanitized = col.replace(" ", "_").replace("/", "_").replace("\\", "_")
    plt.tight_layout(rect=(0, 0.03, 1, 0.95))
    plt.savefig(f"transformation_preview/{col_sanitized}_quantile_comparison.png")
    plt.close()